# Capstone Project - HighCarbon Trip Prediction

## 1. Data Loading

In [1]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

public_df = pd.read_csv('../Public Dataset/public_trip_data.csv')
public_attr = pd.read_csv('../Public Dataset/public_trip_event_attributes.csv')

private_df = pd.read_csv('../Private dataset/private_trip_data.csv')
private_attr = pd.read_csv('../Private dataset/private_trip_event_attributes.csv')

prohibited = ['Departure_CO2e', 'Return_CO2e', 'Hotel_CO2e', 'Spend_CO2e', 'TotalCO2e', 'HighCarbon']
target = 'HighCarbon'
y = public_df[target]

for col in prohibited:
    if col in public_df.columns:
        public_df = public_df.drop(columns=[col])


## 2. Feature Engineering

In [2]:
drop_cols = ['EmployeeNumber', 'TravelRequestID', 'ExpenseRequestID']
for col in drop_cols:
    if col in public_df.columns: public_df = public_df.drop(columns=[col])
    if col in public_attr.columns: public_attr = public_attr.drop(columns=[col])
    if col in private_df.columns: private_df = private_df.drop(columns=[col])
    if col in private_attr.columns: private_attr = private_attr.drop(columns=[col])

train_data = public_df.merge(public_attr, on='TripID', how='left')
test_data = private_df.merge(private_attr, on='TripID', how='left')


## 3. Handling Missing Values

In [3]:
categorical_cols = train_data.select_dtypes(include=['object']).columns.tolist()

for col in categorical_cols:
    train_data[col] = train_data[col].fillna('Missing').astype(str)
    test_data[col] = test_data[col].fillna('Missing').astype(str)

numerical_cols = train_data.select_dtypes(exclude=['object']).columns.tolist()
if 'TripID' in numerical_cols:
    numerical_cols.remove('TripID')

for col in numerical_cols:
    train_data[col] = train_data[col].fillna(0)
    test_data[col] = test_data[col].fillna(0)

features = categorical_cols + numerical_cols
X = train_data[features]
X_test = test_data[features]


## 4. Modeling

In [4]:
cat_features = categorical_cols

params = {
    'iterations': 500,
    'learning_rate': 0.05,
    'depth': 6,
    'eval_metric': 'AUC',
    'random_seed': 42,
    'verbose': 50
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f'--- Fold {fold+1} ---' )
    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]
    
    train_pool = Pool(X_tr, y_tr, cat_features=cat_features)
    val_pool = Pool(X_va, y_va, cat_features=cat_features)
    
    model = CatBoostClassifier(**params)
    model.fit(train_pool, eval_set=val_pool, early_stopping_rounds=100)
    
    oof_preds[val_idx] = model.predict_proba(X_va)[:, 1]
    test_preds += model.predict_proba(X_test)[:, 1] / skf.n_splits

print(f'Overall CV AUC: {roc_auc_score(y, oof_preds):.4f}')


--- Fold 1 ---


0:	test: 0.9986026	best: 0.9986026 (0)	total: 402ms	remaining: 3m 20s


50:	test: 0.9992526	best: 0.9992533 (48)	total: 9.53s	remaining: 1m 23s


100:	test: 0.9992707	best: 0.9992906 (82)	total: 19.1s	remaining: 1m 15s


150:	test: 0.9992683	best: 0.9992965 (116)	total: 28.1s	remaining: 1m 5s


...

50:	test: 0.9991392	best: 0.9991519 (44)	total: 13.9s	remaining: 2m 2s


100:	test: 0.9992725	best: 0.9993055 (85)	total: 25.3s	remaining: 1m 40s


150:	test: 0.9992853	best: 0.9993055 (85)	total: 34.1s	remaining: 1m 18s


Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.999305493
bestIteration = 85

Shrink model to first 86 iterations.


Overall CV AUC: 0.9993


## 5. Submission

In [5]:
submission = pd.DataFrame({
    'TripID': test_data['TripID'],
    'HighCarbon': test_preds
})
submission.to_csv('submission.csv', index=False)
